# Querying Data Files
Sytax for pathing uses backticks not single quotes

## Preparation
Firstly, I'll need to copy in come files from the guide and then run the notebook

In [0]:
%sql
DESCRIBE DATABASE EXTENDED default

In [0]:
%run ../databricks_repo/School-Setup

## Querying JSON Format
Let's review the students folder

In [0]:
files = dbutils.fs.ls(f"{dataset_school}/students-json")
display(files)

In [0]:
%sql
SELECT * FROM json.`${dataset.school}/students-json/export_001.json`

In [0]:
%sql
-- This query will allow us to query an entire directory of JSON files with export_
SELECT * FROM json.`${dataset.school}/students-json/export_*.json`

In [0]:
%sql
-- Even moreso, you can query an entire directory of files (assuming consistent schema and format)
SELECT * FROM json.`${dataset.school}/students-json`

In [0]:
%sql
-- built in Spark SQL functions records the source data file for each record
SELECT *, input_file_name() source_file
FROM json.`${dataset.school}/students-json`;

## Querying Using Text Format
Text format allows an increased functionality to parse a the raw string as you'd like.

In [0]:
%sql
SELECT * FROM text.`${dataset.school}/students-json`

## Querying Using binaryFile format
binary representation may be essential (images, unstructeured data, etc.), and so we can use binaryFile

It also offers content and metadata of files.

In [0]:
%sql
SELECT * FROM binaryFile.`${dataset.school}/students-json`

## Querying Non-Self-Describing Formats
JSON and Parquet formats lend themselves to schemas, CSVs do not and need schema definition to be queried

In [0]:
files = dbutils.fs.ls(f"{dataset_school}")
display(files)

In [0]:
%sql
SELECT * FROM csv.`${dataset.school}/courses-csv`

## Registering Tables from Files with CTAS
Creating tables as well as populating them

In [0]:
%sql
CREATE TABLE students AS
SELECT * FROM json.`${dataset.school}/students-json`;
    
DESCRIBE EXTENDED students;

### Limitations
Support for additional file types is limited as we'll see with CSVs

In [0]:
%sql
CREATE TABLE courses_unparsed AS
SELECT * FROM csv.`${dataset.school}/courses-csv`;

SELECT * FROM courses_unparsed;

## Registering Tables on Foreign Data Sources
USING parameter allows us to deal with formats tha tneed specific configurations
This creates an external table with a manually defined schema.

### Example 1 CSV:
*`CREATE TABLE` csv_external_table <br>
 (col_name1 col_type1, ...) <br>
 `USING` CSV <br>
 `OPTIONS` (header = "true", <br>
           delimiter = ";") <br>
 `LOCATION` = '/path/to/csv/files'* 

 It is dually important to note the initial load and creation solidifies the order of the columns and how they load into the table. If the order of the columns should change on the csv in the future, this will impact the integrity of the data loading process.

In [0]:
%sql
-- let's try with our data
CREATE TABLE courses_csv
  (course_id STRING, title STRING, instructor STRING, category STRING, price DOUBLE)
USING CSV
OPTIONS (header = "true",
        delimiter = ";")
LOCATION "${dataset.school}/courses-csv"

In [0]:
%sql
SELECT * FROM courses_csv

### Example 2 Database
JDBC connection allows referencing data in an external SQL database.

`CREATE TABLE` jdbc_external_table <br>
`USING` JDBC <br>
`OPTIONS` ( <br>
 url = 'jdbc:mysql://your_database_server:port', <br>
 dbtable = 'your_database.table_name',<br>
 user = 'your_username',<br>
 password = 'your_password'<br>
) <br>
`LOCATION` = '/path/to/csv/files'* 

### Limitations
It is also important to note that these foreign sources are not delta tables either.
We'll explore this with our external table linked directly to CSV files.

In [0]:
%sql
DESCRIBE EXTENDED courses_csv

Absense of Delta table (seen in Provider field of DESCRIBE EXTENDED) leaves us with limitations and impacts like outdated data.

In [0]:
files = dbutils.fs.ls(f"{dataset_school}/courses-csv")
display(files)

Let's simulate the ingestion of new CSV files as copy_001.csv

In [0]:
dbutils.fs.cp(f"{dataset_school}/courses-csv/export_001.csv", f"{dataset_school}/courses-csv/copy_001.csv")

In [0]:
display(dbutils.fs.ls(f"{dataset_school}/courses-csv/"))

External CSV file does not natively signal Spark to refresh cached data, we must instruct it to do so.

In [0]:
%sql
REFRESH TABLE courses_csv

It is noteworthy that the csv data must be cached into memory once again which can be time-consuming when dealing with large CSVs

In [0]:
%sql
SELECT count(1) FROM courses_csv

### Hybrid Approach
Creating a temp view that refers to the foreign data source.
Then, execute a CTAS statement on this temp view to extract data and load into Delta table.

In [0]:
%sql
-- Let's simulate creating a delta table for a csv data source
CREATE TEMP VIEW courses_tmp_vw
  (course_id STRING, title STRING, instructor STRING, category STRING, 
  price DOUBLE)
USING CSV
OPTIONS (
 path = "${dataset.school}/courses-csv/export_*.csv",
 header = "true",
 delimiter = ";"
);

CREATE TABLE courses AS
 SELECT * FROM courses_tmp_vw;

In [0]:
%sql
DESCRIBE EXTENDED courses

In [0]:
%sql
SELECT * FROM courses

# Writing to Tables

## Preparation
Make sure to run School-Setup again to prepare our environment, if needed.
Then, we'll run a CTAS to make enrollments.

In [0]:
%sql
CREATE TABLE enrollments AS
SELECT * FROM parquet.`${dataset.school}/enrollments`

In [0]:
%sql
SELECT * FROM enrollments

## Replacing Data
Drop and recreate vs Overwrite<br>
Overwrite has many advantages.
It provides efficiency, reliability, and seamless integration with Delta's features such as time travel and ACID transactions.<br>
<br>
Two methods to completely replace the content of Delta Lake tables:
- `CREATE OR REPLACE TABLE` aka CRAS
- `INSERT OVERWRITE`

### CRAS

In [0]:
%sql
-- This query will overwrite the table with newer data
CREATE OR REPLACE TABLE enrollments AS
SELECT * FROM parquet.`${dataset.school}/enrollments`

In [0]:
%sql
-- Let's check out the table history to see what happened
DESCRIBE HISTORY enrollments

### Insert or Overwrite
Unlike CRAS, INSERT OVERWRITE requires a table to exist.

In [0]:
%sql
INSERT OVERWRITE enrollments
SELECT * FROM parquet.`${dataset.school}/enrollments`

Categorized as a WRITE operation with mode Overwrite

In [0]:
%sql
DESCRIBE HISTORY enrollments

A significant advantage of using INSERT OVERWRITE is its ability to overwrite only the new records tha tmatch the current table schema. This prevents risk of modifying the table structure. Thus, is considered more secure for overwriting existing tables. Mismatches in schema will throw an error.

In [0]:
%sql
INSERT OVERWRITE enrollments
SELECT *, input_file_name() FROM parquet.`${dataset.school}/enrollments`

## Appending Data

In [0]:
%run ./School-Setup

In [0]:
%sql
INSERT INTO enrollments
SELECT * FROM parquet.`${dataset.school}/enrollments-new`

In [0]:
%sql
SELECT count(1) FROM enrollments

## Merging Data

While convenient, it does lack the ability to prevent duplicate data.
That's where MERGE INTO comes into play.<br>
<br>
Let's view this operation by updating student data with modified email addresses and add new students into the table. <br>
<br>
First, create a temp view containing updated student data. This will serve as source from which we'll merge changes into our students table.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW students_updates AS
SELECT * FROM json.`${dataset.school}/students-json-new`;

Next we'll actually use the MERGE INTO on the student_id to update the email or insert a new student if there's no match on our student database


In [0]:
%sql
MERGE INTO students c
USING students_updates u
ON c.student_id = u.student_id
WHEN MATCHED AND c.email IS NULL AND u.email IS NOT NULL THEN
  UPDATE SET email = u.email, updated = u.updated
WHEN NOT MATCHED THEN
  INSERT *

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW courses_updates
  (course_id STRING, title STRING, instructor STRING, category STRING, price DOUBLE)
USING CSV
OPTIONS (
  header = "true",
  delimiter = ";",
  path = "${dataset.school}/courses-csv-new"
);


Focusing exclusively on inserts here, we only insert based on unique key comprising course_id and title fields

In [0]:
%sql
MERGE INTO courses c
USING courses_updates u
ON c.course_id = u.course_id
WHEN NOT MATCHED AND u.category = 'Computer Science' THEN
  INSERT *

This is an insert-only merge- this demonstrates the primary advantages of the merge operation, the ability to prevent duplicate entries.

# Performing Advanced ETL Transformations

We will investigate methods of traversing nested and complex data structures.

In [0]:
%run ./School-Setup

In [0]:
%sql
SELECT * FROM students

We can see the profile is a JSON object, let's look at the schema of the table

In [0]:
%sql
DESCRIBE students

Profile is a string, but its also a JSON string- enabling us to use colon syntax

In [0]:
%sql
SELECT  student_id,
        profile:first_name,
        profile:address:country
FROM students

We are also afforded the functionality to parse JSON object into struct types, but requires knowledge of the schema prior.

This would allow us to parse that exactly, details to follow.
`SELECT from_json(profile, <schema>) FROM students;`

In [0]:
%sql
-- Let's create a view to parse the JSON column
CREATE OR REPLACE TEMP VIEW parsed_students AS
 SELECT student_id, from_json(profile, schema_of_json('{"first_name":"Sarah",
 "last_name":"Lundi", "gender":"Female", "address":{"street":"8 Greenbank Road",
 "city":"Ottawa", "country":"Canada"}}')) AS profile_struct
 FROM students;

SELECT * FROM parsed_students

In [0]:
%sql
DESCRIBE parsed_students

In [0]:
%sql
-- Struct allows us to query using dot notation 
SELECT student_id, profile_struct.first_name, profile_struct.address.country
FROM parsed_students

In [0]:
%sql
-- Furthermore, we can flatten fields and create separate columns with STAR operator
CREATE OR REPLACE TEMP VIEW students_final AS
SELECT student_id, profile_struct.*
FROM parsed_students;

SELECT * FROM students_final

### Leveraging the explode Function

In [0]:
%sql
SELECT enroll_id, student_id, courses
FROM enrollments

The courses column is an array of structs. Explode allows us to transform an array into individual rows each representing an element in the array.

In [0]:
%sql
SELECT enroll_id, student_id, explode(courses) AS course
FROM enrollments

Any new records created with explode duplicate the enroll_id and student_id information over into the newly created records.

### Aggregating Unique Values

In [0]:
%sql
-- Let's investigate the collect_set function that returns an array of unique values for a given field
SELECT student_id,
       collect_set(enroll_id) AS enrollment_set,
       collect_set(courses.course_id) AS courses_set
FROM enrollments
GROUP BY student_id

However, courses_set is an array of arrays which could have have dupliate values across arrays, since its only making a set of each array and not the entire array. We can flatten the array using `flatten` and then use `array_distinct` function to retain on the unique values.

In [0]:
%sql
-- Ok, so we need to flatten and then use array_distinct 
SELECT student_id,
  collect_set(courses.course_id) AS before_flatten,
  array_distinct(flatten(collect_set(courses.course_id))) AS after_flatten
FROM enrollments
GROUP BY student_id

## Mastering Join Operations in Spark SQL
Spark SQL supports all the traditional joins inner, outer, left, right, cross, anti, and semi

In [0]:
%sql
CREATE OR REPLACE VIEW enrollments_enriched AS
SELECT *
FROM (
  SELECT *, explode(courses) AS course
  FROM enrollments
) e
INNER JOIN 
  courses c ON c.course_id = e.course.course_id;

SELECT * FROM enrollments_enriched;

## Exploring Set Operations in Spark SQL

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW enrollments_updates AS
SELECT * FROM parquet.`${dataset.school}/enrollments-new`;

In [0]:
%sql
SELECT * FROM enrollments
UNION ALL
SELECT * FROM enrollments_updates

In [0]:
%sql
SELECT * FROM enrollments
INTERSECT
SELECT * FROM enrollments_updates

In [0]:
%sql
-- The MINUS operation allows us to obtain records exclusive to one 
SELECT * FROM enrollments
MINUS
SELECT * FROM enrollments_updates

## Changing Data Perspectives
Spark SQL supports creating pivot tables for transforming data perspectives using PIVOT

In [0]:
%sql
SELECT * FROM (
  SELECT student_id, course.course_id AS course_id, course.subtotal AS subtotal
  FROM enrollments_enriched
)
PIVOT (
  SUM(subtotal) FOR course_id IN (
    'C01', 'C02', 'C03', 'C04', 'C05', 'C06',
    'C07', 'C08', 'C09', 'C10', 'C11', 'C12'
  )
)

# Working with Higher-Order Functions

Higher-order functions provide powerful toolset for working with complex data types. 
Let's cover:
- FILTER
- TRANSFORM

In [0]:
%run ./School-Setup

In [0]:
%sql
SELECT * FROM enrollments

## Filter Function
FILTER is a fundamental higher-order function that enables the extraction of specific elements from an array based on a given lambda function.

In [0]:
%sql
-- Let's use FILTER to create a new column called highly_discounted_courses
SELECT enroll_id,
       courses,
       FILTER (courses, c -> c.discount_percent >= 60) AS highly_discounted_courses
FROM enrollments

In [0]:
%sql
-- We have to make a subquery to filter out the arrays that are empty (size > 0)
SELECT enroll_id, highly_discounted_courses
FROM (
  SELECT enroll_id,
         courses,
         FILTER (courses, c -> c.discount_percent >= 60) AS highly_discounted_courses
  FROM enrollments)
WHERE size(highly_discounted_courses) > 0;

## Transform Function
TRANSFORM function is another higher-order function.

In [0]:
%sql
SELECT enroll_id,
       courses,
       TRANSFORM(
        courses,
        c -> ROUND(c.subtotal * 1.2, 2)
       ) AS courses_after_tax
FROM enrollments


In [0]:
%sql
SELECT enroll_id,
       courses,
       TRANSFORM(
        courses,
        c -> (c.course_id, ROUND(c.subtotal * 1.2, 2) AS sub_total_with_tax)
       ) AS courses_after_tax
FROM enrollments


# Developing SQL UDFs

SQL User-defined Functions (UDFs) can encapsulate custom logic with SQL-like syntax.

## Creating UDFs
Let's examine this functionality by converting student's GPA into a percentage equivalent. It will multiply GPA by 25 and then rounded to nearest integer

In [0]:
%sql
CREATE OR REPLACE FUNCTION gpa_to_percentage(gpa DOUBLE)
RETURNS INT
RETURN CAST(ROUND(gpa * 25) AS INT)

## Applying UDFs

In [0]:
%sql
SELECT student_id, gpa, gpa_to_percentage(gpa) AS percentage_score
FROM students

## Understanding UDFs
SQL UDFs are permanent object stored in the database.

In [0]:
%sql
DESCRIBE FUNCTION gpa_to_percentage

In [0]:
%sql
DESCRIBE FUNCTION EXTENDED gpa_to_percentage

In [0]:
%sql
CREATE OR REPLACE FUNCTION get_letter_grade(gpa DOUBLE)
RETURNS STRING
RETURN CASE
  WHEN gpa >= 3.5 THEN "A"
  WHEN gpa >= 2.75 AND gpa < 3.5 THEN "B"
  WHEN gpa >= 2.0 AND gpa < 2.75 THEN "C"
  ELSE "F"
END;
    
SELECT student_id, gpa, get_letter_grade(gpa) AS letter_grade
FROM students

## Dropping UDFs

In [0]:
%sql
DROP FUNCTION gpa_to_percentage;
DROP FUNCTION get_letter_grade;